# Dropout Optimizations
![Example of Dropout](../assets/DropoutExample.png)
* **Philox PRNG Architecture**: Salmon, J. K., Moraes, M. A., Dror, R. O., & Shaw, D. E. (2011). Parallel Random Numbers: As Easy as 1, 2, 3. In Proceedings of the International Conference for High Performance Computing, Networking, Storage and Analysis (pp. 1-12).
* **PyTorch PRNG Walkthrough:** Deshpande, A. (2023). *How PyTorch Generates Random Numbers*. Coding Confessions. https://blog.codingconfessions.com/p/how-pytorch-generates-random-numbers

This notebook draws theoretical concepts, constants, and round permutations from the original paper above, alongside practical insights on PyTorch's internal C++ implementation from Coding Confessions. Aside from `SpatialDropout`,`Dropout` is the only layer in this framework that implements an advanced custom kernel. While rewriting every layer at this level of hardware optimization is beyond the scope of this project, these two implementations serve as a targeted deep dive into high performance CUDA/HIP/C++ operations. 
The old implementation is shown below: 

In [ ]:
try:
    import cupy as cp
    xp = cp
except (ModuleNotFoundError, ImportError):
    import numpy as np
    xp = np 

class Dropout:
    def __init__(self, rate):
        #We write rate as the success rate. The dropout rate will then be 
        self.rate = 1 - rate
    
    def forward(self, inputs, training):
        #were gonna save the inputs and the binary mask
        self.inputs = inputs
        if not training:
            self.output = inputs.copy()
            return self.output
        self.binary_mask = xp.random.binomial(1, self.rate, size = inputs.shape) \
                        / self.rate
        self.output = self.binary_mask * self.inputs

        return self.output
    
    def backward(self, dvalues):
        self.dinputs = dvalues * self.binary_mask 

## Can we Really make any Improvements for Dropout? 

In general, with GPU programming there are two types of bounds to account for. 

* **Compute Bound**: Where the amount of FLOPs(floating point operations) of a compute core are saturated, meaning data has to wait for a cores thread to complete its task before moving on. 
* **Memory Bound**: Where the compute cores are waiting for data to be fetched from GPU VRAM. In this case, while GPUs benefit from holding information inside L1 and L2 cache, and perform calculations using the GPU registers. Lower level memory types can't account for slow VRAM, meaning compute cores idle waiting for new work. 

Our solution to memory bound, or more specifically memory bandwidth problems come from **kernel fusion** (`@cp.fuse()`, `cp.ElementwiseKernel`):  
  
We can circumvent this by avoiding unnecessary kernel operations, oftentimes combining operations (otherwise combining kernels) into a single or custom kernel that performs multiple operations at the same time. 
* A fused kernel will have data read from VRAM once, with operations being performed on the fly with the faster GPU registers and cache. Finally, we create a final tensor **once**, and write this back onto GPU VRAM. 

In our old implementation, we required 4 seperate kernel operations. Assume N elements means a 4D tensor:

    Step 1: tmp1 = xp.random.binomial(1, ...)                   ──► Launches Kernel 1 (Allocates N elements for tmp1)  
    Step 2: self.binary_mask = tmp1 / self.rate                 ──► Launches Kernel 2 (Allocates N elements for binary_mask)  
    Step 3: self.output = self.binary_mask * self.inputs        ──► Launches Kernel 3 (Allocates N elements for output)
This means we read from VRAM 3 times, and write to VRAM 3 times. We'll rewrite this pass such that we read from VRAM once, and write to VRAM once. We will be implementing the following:

    Step 1: self.output = _fused_dropout(inputs, self.rate)     ──► Launches Kernel 1 (Allocates N elements ONLY for output)

Normally I would discuss the use of the decorator `@cp.fuse()` which will combine out of place operations **or** out of place reductions into a fused kernel. However, for this case we'll use the `cp.ElementwiseKernel()` to create a custom kernel and create this fused dropout layer. For this, although we have a division by a scaler, then an elementwise multiplication operation after, we cannot combine these operations with `cp.random.binomial` as this is **not** a universal function. To recap, `@cp.fuse()` works with universal functions only (+, -, *, /) or `cp.add()`, `cp.multiply()`, `cp.maximum`, etc. in either elementwise or reduction operations, but does not work outside of these bounds. 



# How will we Code a Custom Kernel then? 
Looking at our code, we want to create a binary mask (a tensor of only 0s and 1s) and divide this by our rate `self.rate`. The simple reason for this is to amplify the remaining neurons such that the expected value of the tensor remains the same. Next, we'll perform an elementwise multiplication between the binary mask and our inputs, resulting in our desired `self.output`. 



The hard part will be creating our own `cp.random.binomial()` function that will we'll apply onto our inputs. The division and elementwise multiplication steps are trivial. 

Looking at how the other deep learning frameworks write their dropout layer, they avoid a costly random generation by using **Philox4x32-10 PRNG**, a counter-based psudo random number generator that uses an integer counter for the inital state along with mulitplication based mixing. Normal PRNGs mutate an internal state sequentially, ($S_{t+1} = f(S_t))$, which is costly for a GPU.

### How does Philox Work? 

For **Philox-4x32**, we'll contain 4 32-bit integers at a time. We'll have a 128-bit counter (GPUs read from VRAM in 128 bit contigous blocks per memory transaction), and produces a 128-bit output (four 32-bit random numbers). This means we generate 4 random numbers at once per memory transaction. There are 10 rounds with an order shown below. 

* Four 32-bit words  $[c_0, c_1, c_2, c_3]$.
$$
\text{128-Bit Counter} = [\ \underbrace{c_3}_{\text{bits 96..127}}\ \vert\ \underbrace{c_2}_{\text{bits 64..95}}\ \vert\ \underbrace{c_1}_{\text{bits 32..63}}\ \vert\ \underbrace{c_0}_{\text{bits 0..31}}\ ]
$$
* Key (K): Two 32-bit words $[K_0, K_1]$.
* Multipliers ($M_0, M_1$): Hardcoded 32 bit integers that satisfied high spectral test scores, with no short bit patterns, and odd hex digits. They do their job, and they do their job well. 
$$M_0 = \text{0xD2511F53}, \quad M_1 = \text{0xCD9E8D57}$$

Each round we perform 2 multiplications: 
$$
    \text{prod}_0 = M_0 \times c_0, \quad \text{prod}_1 = M_1 \times c_2
$$
With this, we arrive at two 64-bit results (prod). Inside each prod, we split the bit values into **High 32 bits** (the bits on the right), and **Low 32 bits** (the bits on the left). Next, the new state words for the next round ($c'$) are computed by combining the product halves with the state word and the key. 
$$
h_0 = \text{hi}(\text{prod}_0)) \oplus c_1 \oplus K_0
$$
$$
h_1 = \text{hi}(\text{prod}_1)) \oplus c_3 \oplus K_1
$$

Now we can rearrange the values for the next round. The output for a round become
$$
(c'_0, c'_1, c'_2, c'_3) = \left( \text{hi}_1 \oplus c_1 \oplus K_0, \; \text{lo}_1, \; \text{hi}_0 \oplus c_3 \oplus K_1, \; \text{lo}_0 \right)
$$
The values have now been shuffled around, the low parts of the products arrive at positions 0 and 2, while the XORed high parts sit at position 1 and 3.

#### After Each Round

Before the final 10th round, we'll update the keys themselves by two constants 

$$
K'_0 = K_0 + w_0, \quad K'_1 = K_1 + w_1
$$
where:
* $w_0$: 0x9E3779B9
* $w_1$: 0xBB67AE85
From the paper, these are the Weyl sequence constants. 
For those who require a diagram, it is shown below:  
  
![alt text](../assets/PhiloxDiagram.png)

# Implementation of Dropout using Philox PRNG 

We know we're going to need the algorithm for Philox during implementation. However, we'll create a helper function `_PHILOX_PREAMBLE` which we'll use for the forward and backward passes of `Dropout` and `Spatial_Dropout`. We'll worry about philox last. Lets first worry about creating forward and backward passes for Dropout. 

If we envoke `_PHILOX_PREAMBLE` using the same seed `philox_seed` and the same offset `philox_offset`, then we'll always arrive at the same random values. This is an issue if we want to have randomized values for each instance of a layer. Each time a layers class is called (in this case either of the dropout classes) then we'll have to use a custom function. In this case, `np.random.SeedSequence()` is perfect for generating a truly random seed!

* `np.random.SeedSequence()`: Creates a hash function (MixMax/PCG) to mix `base_seed` and a `stream_id` (instance of a layer). This results in an 64-bit unisgned integer that we'll pass into our `_PHILOX_PREAMBLE` when we get to that. We'll use two methods for this 
    * `entropy`: The entropy for creating a seed.
    * `spawn_key`: An aditional source of entropy, which we'll pass in our `_stream_counter` as this. 

We'll create a helper function for this branch of code in our init method (this will be moved to the base class `Layer`):
```python
class Layer():
    def __init__(self, seed):
        self.seed = seed

    def _derive_stream_seed(base_seed, stream_id):
        if base_seed is None: 
            entropy = None
            spawn_key = (int(stream_id))
        else:
            entropy = [int(base_seed), int(stream_id)]
            spawn_key = ()
        
        seed_seq = np.random.SeedSequence(entropy, spawn_key=spawn_key)
        return int(seed_seq.generate_state(1, dtype=np.uint64[0]))

class Dropout(Layer):
    # each dropout instance recieves a unique stream ID
    _stream_counter = 0
    def __init__(self, rate, seed=None) #this seed can be passed in from Model later on
        super().__init__(seed=seed) # from Layer.__init__
        self.rate = 1 - rate

        stream_id = Dropout._stream_counter
        Dropout._stream_counter += 1
        self._seed = _derive_stream_seed(self.seed, stream_id)
        self._call_counter = 0 #our c0, c1 for philox
```

We now have a custom 64 bit seed that we'll pass into the Philox function. Given the `__init__` method, we can now create a GPU branch for the forward pass. Here, we'll check if we are training or not, update the `self._call_counter`, and call our elementwise kernel. Since we need to keep a copy of the offset for our backwards pass, we'll save that as well.

```python
def _forward_gpu(self, inputs, training)
    if not training:
        self.output = inputs
    
    self._call_counter += 1 
    offset = self._call_counter
    self._offset = offset

    self.output = _philox_dropout_forward(
        inputs, self._seed, offset, float(self.rate)
    )
    return self.output
```

### Creation of `_philox_dropout_forward`

This is where we'll combine the old operations of combining the inputs and binary mask while dividing that by our probability `self.rate`. With `cp.ElementwiseKernel()`, CuPy will let us write C/C++ code inside python string blocks. In these languages, we have to specify our datatype during creation, since we passed in four arguments above we'll list the parameters. The arguments must be in a specific order:
1. Input Arguments (`in_params` string)
2. Output Arguments (`out_params` string)
3. Kernel Body Code (`operation` string)
4. Kernel Name (`name` string)
5. Additional Operations, (we'll use preamble here) 

```python
_philox_dropout_forward = cp.ElementwiseKernel(
    # 1, we'll list the inputs and outputs 
    'T x, uint64 philox_seed, uint64 philox_offset, float64 keep_prob',
    # 2
    'T y',
    # 3...
    '''
    # operation goes here
    '''
```
We'll pretend that we already have created our preamble function that will generate the 32bit random number. This can be set as `float u`. Now, we'll call this helper function `philox_uniform()` that requires the entropy arguments along with the array element index. In CuPy, `i` is a special variable that is a pointer that points to the index number of the current array element being processed by a specific thread (Thread 0 gets `i = 0`, Thread 1 gets `i = 1`). 
```python
float u = philox_uniform(philox_seed, philox_offset, i)
```
Now that we have our *u* value, we can create a line that calculates for the final element at a specific index. We can use a standard if/else staement but we'll write a ternary operation instead `(condition ? true_value : false_value)`. We'll use a type case `(T)` such that we cast the value as `float32` rather then risking a calculation being saved as `flaot64`. The full forward pass will be below

```python

_philox_dropout_forward = cp.ElementwiseKernel(
' T x, uint64 philox_seed, uint64 philox_offset, float64 keep_prob',
' T y', 
'''
u = philox_uniform(philox_seed, philox_offset, i);
y = (u < keep_prob) ? (T)(x / keep_prob) : (T)(0);
''',
'philox_dropout_forward', 
preamble=_PHILOX_PREAMBLE,
)
```

## Creation of `philox_dropout_backward`

Before we create our complicated helper function, we can easily rewrite the backward pass such that we retrieve our old randomly generated output value. The savings are real as this means we can avoid saving some copy of `self.binary_mask` which was in the old implementation. 

Our backwards pass before was `self.dinputs = dvalues * self.binary_mask`. We'll create a second kernel that recreates the old binary_mask value, then we'll do a simple multiplication. It will mean we'll recrate the old forward pass in a sense, but the computational cost is still so low compared to the memory cost and memory bandwidth constraints it is still a worthwhile investment. 

```python
def backward_gpu(self, dvalues)
    self.dinputs = _philox_dropout_backward(dvalues, self._seed, self._offset, float(self.rate))
    return self.dinputs

# now for the kernel that uses (condition ? true_value : false_value)

_philox_dropout_backward = cp.ElementwiseKernel(
    'T dvalues, uint64 philox_seed, uint64 philox_offset, float64 keep_prob', 
    'T dinputs', 
    '''
    float u = philox_uniform(philox_seed, philox_offset, i);
    dinputs = (u < keep_prob) ? (T)(dvalues / keep_prob) : (T)(0);
    '''
    '_philox_dropout_backward', 
    preamble = _PHILOX_PREAMBLE
    )
```

If you have been following along in this notebook, you may notice that the code itself is not hard to follow! The preamble may be a different story though...

# Implementation of _PHILOX_PREAMBLE = philox_uniform(philox_seed, philox_offset, i) 

A few constants we need, the two multiplication constants along with the two Weyl constants needed to update the keys. 
$$M_0 = \text{0xD2511F53}, \quad M_1 = \text{0xCD9E8D57}, \quad w_0 = \text{0x9E3779B9}, w_1 = \text{0XBB67AE85}$$

The algorithm is a simple for loop that will perform the operations shown in the notes and diagram. Since we know what sizes each of the variables are, we'll start with that. 

We need to declare a few things, (else how is it supposed to know we're runing it in a GPU or to skip function-call overhead). 
* `usigned long long`: 64 bits that store only non-negative values
* `unsigned long`: Usually 32 bits that will store only non-negative values
128 bit ALUs don't exist in coding, but we can workaround that by assigning four different 32 bit variables that will act as our 32 bit counter (`c0`, `c1`, `c2`, `c3`):
Now in the *CodingConfessions* article, the upper 64 bits are the thread region we're in, and the lower 64 bits are the position within a threads assigned region, or in other words the offset we've passed into the preamble. 

$$
\begin{matrix}
\begin{array}{c|c||c|c}
c_3 & c_2 & c_1 & c_0 \\
\text{Bits 96..127} & \text{Bits 64..95} & \text{Bits 32..63} & \text{Bits 0..31} \\
\hline
\text{Upper 32 (idx)} & \text{Lower 32 (idx)} & \text{Upper 32 (offset)} & \text{Lower 32 (offset)}
\end{array} \\
\underbrace{\hspace{8.5em}}_{\text{Upper 64 Bits (Thread idx)}} \quad \underbrace{\hspace{9em}}_{\text{Lower 64 Bits (Batch offset)}}
\end{matrix}
$$

#### How will we split a 64 bit integer into two 32 bit integers? 

Apprarently we have to use bit masking for the upper 32 bits, and bit shifting for the lower 32 bits. 

For the upper bits, imagine we our philox offset already defined. 
$$
\mathtt{philox\_offset} = \underbrace{\mathtt{0x12345678}}_{\text{Upper 32 Bits}} \quad \underbrace{\mathtt{0x9ABCDEF0}}_{\text{Lower 32 Bits}}
$$

If we want to get the lower 32 bits, (our `c0` and `c2`), we need to find a way to get rid of the upper 32 bits, while still preserving the data of the lower 32 bits. 
In hexidecimal, each character corresponds to 4 bits, so a character $f$ represents `15` in decimal or `1111` in binary. That means, given the prefix `0x` (for hexadecimal (Base-16) literal), we can write a 32 bit integer of ones as such: 
```c
lower_32_bit_mask = 0xffffffffUL
```
The thing is we still want this to a 64 bit integer to compare our values to, so we can write it as a `ULL` suffix for `unsigned long long`. 
```c
lower_32_bit_mask = 0xffffffffULL
# The same as writing 0x00000000ffffffff
```
Now, we can do a bitwise **AND** operation, whenever both values equal 1, then it is one, otherwise we set to 0. This means we match our lower 32 bits always
```c
# truncate the leading 32 zeros
unsigned int c0 = (unsigned int)(philox_offset & 0xffffffffULL)
```
$$
\begin{array}{rll}   \mathtt{philox\_offset}: & \mathtt{0x12345678\ 9ABCDEF0} & \text{(64-bit value)} \\   \mathtt{\& \ 0xffffffffULL}: & \mathtt{0x00000000\ FFFFFFFF} & \text{(64-bit mask)} \\   \hline   \mathtt{Result}: & \mathtt{0x00000000\ 9ABCDEF0} & \text{(Upper 32 bits wiped out)} \end{array}
$$

Bitshifting is an easier concept to grasp. We can use the `>>` operator followed by the argument `num` which tells the compiler to shift a set of bits by `num` times to the right. We can also do the same to our 64 bit key `philox_seed`Our starting code becomes: 
'''python
_PHILOX_PREAMBLE = r'''
__device__ __forceinline__ float philox_uniform(
    unsigned long long philox_seed, 
    unsigned long long philox_offset, 
    long long idx) {

    unsigned int c0 = (unsigned int(philox_offset & 0xffffffffULL));
    unsigned int c1 = (unsigned int(philox_offset >> 32));
    unsigned int c2 = (unsigned int(idx & 0xffffffffULL));
    unsigned int c3 = (unsigned int(idx >> 32)); 

    unsigned int k0 = (unsigned int(philox_seed & 0xffffffffULL));
    unisgned int k1 = (unsigned int(philox_seed & 0xffffffffULL));
    }

Now we can create the actual for loop, we won't save the constants as variables instead we'll use the hardcoded values in place (for a little more performance). Since the algorithm is already made, we can code it directly below (note xor operation in c is ^):

```python
#pragma unroll 
for (int round = 0, round < 10, round++){
    unsigned long long p0 = (unsigned long long) 0xD2511F53u * c0;
    unsigned long long p1 = (unsigned long long) 0xCD9E8D57u * c2;
    
    unsigned int hi0 = (unsigned int) (p0 >> 32); 
    unsigned int lo0 = (unsigned int) (p0 & 0xffffffffULL);
    unsigned int hi1 = (unsigned int) (p1 >> 32);
    unsigned int lo1 = (unsigned int) (p1 & 0xffffffffULL);

    unsigned int nc0 = hi1 ^ c1 ^ k0
    unsigned int nc1 = lo1
    unsigned int nc2 = hi0 ^ c3 ^ k1
    unsigned int nc3 = lo0

    c0 = nc0; c1 = nc1; c2 = nc2; c3 = nc3; 
    k0 += 0x9E3779B9u
    k1 += 0xBB67AE85u
    }
    return (c0 >> 8) * (1.0f / 16777216.0f);
```

We have no completed creating a custom gpu path for both forward and backward passes, along with creating two elementwise kernels for the respective layers that call on the preamble `philox_uniform`.